In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys

os.environ["TORCHDYNAMO_INLINE_INBUILT_NN_MODULES"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
if sys.platform != "darwin":
    os.environ["MUJOCO_GL"] = "egl"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_DEFAULT_MATMUL_PRECISION"] = "highest"


import random
import time

import tqdm
import wandb
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler
from tensordict import TensorDict, from_module

torch.autograd.set_detect_anomaly(True)
torch.set_float32_matmul_precision("high")

from fast_td3.fast_td3_utils import (
    EmpiricalNormalization,
    SimpleReplayBufferGNN,
    save_params,
)

from fast_td3 import Critic
from fast_td3.actors import ActorEGNN, Actor, ActorMPNN

In [ ]:
from fast_td3.hyperparams import HumanoidBenchArgs

robot = "h1"

args = HumanoidBenchArgs(
    env_name=f"{robot}-stand-v0",
    total_timesteps=50000,
    render_interval=5000,
    eval_interval=1000,
    num_envs=16,
    batch_size=8192
)

In [ ]:
amp_enabled = args.amp and args.cuda and torch.cuda.is_available()
amp_device_type = (
    "cuda"
    if args.cuda and torch.cuda.is_available()
    else "mps" if args.cuda and torch.backends.mps.is_available() else "cpu"
)
amp_dtype = torch.bfloat16 if args.amp_dtype == "bf16" else torch.float16

scaler = GradScaler(enabled=amp_enabled and amp_dtype == torch.float16)


random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)
torch.backends.cudnn.deterministic = args.torch_deterministic

if not args.cuda:
    device = torch.device("cpu")
else:
    if torch.cuda.is_available():
        device = torch.device(f"cuda:{args.device_rank}")
    elif torch.backends.mps.is_available():
        device = torch.device(f"mps:{args.device_rank}")
    else:
        raise ValueError("No GPU available")
print(f"Using device: {device}")

In [ ]:
from fast_td3.environments.humanoid_bench_env import HumanoidBenchEnv

env_type = "humanoid_bench"
envs = HumanoidBenchEnv(args.env_name, args.num_envs, device=device)
eval_envs = envs
render_env = HumanoidBenchEnv(args.env_name, 1, render_mode="rgb_array", device=device)

n_act = envs.num_actions
n_obs = envs.num_obs if type(envs.num_obs) == int else envs.num_obs[0]
if envs.asymmetric_obs:
    n_critic_obs = (
        envs.num_privileged_obs
        if type(envs.num_privileged_obs) == int
        else envs.num_privileged_obs[0]
    )
else:
    n_critic_obs = n_obs
action_low, action_high = -1.0, 1.0

In [ ]:
checkpoint_path = None
checkpoint_path = "./models/egnn h1-stand-v0 16envs 100000steps 2025-07-04 17:42:38_final.pt"
obs_normalizer = EmpiricalNormalization(shape=n_obs, device=device)
xpos_normalizer = EmpiricalNormalization(shape=(20, 3), device=device)
normalize_obs = obs_normalizer.forward
normalize_xpos = xpos_normalizer.forward

# Actor setup
actor = ActorEGNN(
    n_obs=n_obs,
    n_act=n_act,
    num_envs=args.num_envs,
    batch_size=args.batch_size,
    device=device,
    init_scale=args.init_scale,
    hidden_dim=80,
    n_layers=4,
    act_fn="relu",
    robot=robot,
)

print("actor parameters:", sum(p.numel() for p in actor.parameters()))

In [ ]:
# checkpoint_path = "./models/mpnn h1-stand-v0 16envs 100000steps 2025-07-03 01:47:04_55000.pt"
# obs_normalizer = EmpiricalNormalization(shape=n_obs, device=device)
# xpos_normalizer = EmpiricalNormalization(shape=(20, 3), device=device)
# normalize_obs = obs_normalizer.forward
# normalize_xpos = xpos_normalizer.forward

# actor = ActorMPNN(
#     n_obs=n_obs,
#     n_act=n_act,
#     num_envs=args.num_envs,
#     batch_size=args.batch_size,
#     device=device,
#     init_scale=args.init_scale,
#     hidden_dim=256,
#     latent_dim=96,
#     n_layers=4,
#     act_fn="relu",
#     robot=robot,
# )

# print("actor parameters:", sum(p.numel() for p in actor.parameters()))

In [ ]:
def evaluate():
    obs_normalizer.eval()
    xpos_normalizer.eval()
    num_eval_envs = eval_envs.num_envs
    episode_returns = torch.zeros(num_eval_envs, device=device)
    episode_lengths = torch.zeros(num_eval_envs, device=device)
    done_masks = torch.zeros(num_eval_envs, dtype=torch.bool, device=device)
   

    
    obs, xpos = eval_envs.reset()

    # Run for a fixed number of steps
    for _ in range(eval_envs.max_episode_steps):
        with torch.no_grad(), autocast(
            device_type=amp_device_type, dtype=amp_dtype, enabled=amp_enabled
        ):  
            # obs = normalize_obs(obs)
            # xpos = normalize_xpos(xpos)
            actions = actor(obs, xpos)

        next_obs, rewards, dones, _ , next_xpos = eval_envs.step(actions.float())
        episode_returns = torch.where(
            ~done_masks, episode_returns + rewards, episode_returns
        )
        episode_lengths = torch.where(~done_masks, episode_lengths + 1, episode_lengths)
        done_masks = torch.logical_or(done_masks, dones)
        if done_masks.all():
            break
        obs = next_obs
        xpos = next_xpos

    obs_normalizer.train()
    xpos_normalizer.train()
    return episode_returns.mean().item(), episode_lengths.mean().item()

In [ ]:
if checkpoint_path is not None:
    torch_checkpoint = torch.load(
        f"{checkpoint_path}", map_location=device, weights_only=False
    )
    # obs_normalizer.load_state_dict(torch_checkpoint["obs_normalizer_state"])
    # xpos_normalizer.load_state_dict(torch_checkpoint["xpos_normalizer_state"])
    pretrained_state_dict = torch_checkpoint["actor_state_dict"]
    for k in list(pretrained_state_dict.keys()):
        if "egnn.edge_index" in k or "egnn.edge_attr" in k:
            del pretrained_state_dict[k]
    
    actor.load_state_dict(torch_checkpoint["actor_state_dict"])
  
evaluate()